# 07 - Modelo de Seguro Complepleto

Este notebook integra los artefactos finales del proyecto para generar una cotización operativa del seguro indexado de café.

**Entradas del usuario**

- `municipio`
- `anio`
- `area_ha`

**Salidas esperadas**

- rendimiento esperado
- producción esperada
- índice de riesgo
- nivel de riesgo
- probabilidad de pérdida
- monto asegurado
- costo de prima
- indemnización estimada
- payout esperado

Este notebook está pensado como puente entre los notebooks de modelado y la futura API.


## 1. Librerías

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 2. Rutas

In [3]:
# ============================================================
# RUTAS ROBUSTAS DEL PROYECTO
# ============================================================

PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

PATH_MODEL = PROJECT_ROOT / "data" / "model"
PATH_OUTPUTS = PROJECT_ROOT / "data" / "outputs"

PATH_OUTPUTS.mkdir(parents=True, exist_ok=True)

# Insumos principales del modelo
FILE_FEATURES_HIST = PATH_MODEL / "features_intra_anuales_2007-2024_clusters.csv"
FILE_FEATURES_FUT = PATH_MODEL / "features_intra_anuales_2025-2027.csv"
FILE_CLUSTERS = PATH_MODEL / "clusters_municipales.csv"
FILE_COEF = PATH_MODEL / "coeficientes_indice_por_cluster.csv"
FILE_VARIABLES = PATH_MODEL / "variables_indice_final.csv"
FILE_PARAMETROS = PATH_MODEL / "parametros_seguro.csv"
FILE_PRECIOS = PATH_MODEL / "precios_cafe_anual_2007-2027.csv"

# Salida
OUTPUT_COTIZACIONES = PATH_OUTPUTS / "cotizaciones_modelo_seguro.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PATH_MODEL:", PATH_MODEL)


PROJECT_ROOT: /home/ddayann/proyectos/Coffe/proyecto_aplicado_en_analitica_de_datos
PATH_MODEL: /home/ddayann/proyectos/Coffe/proyecto_aplicado_en_analitica_de_datos/data/model


## 3. Cargar insumos

In [4]:
features_hist = pd.read_csv(FILE_FEATURES_HIST)
features_fut = pd.read_csv(FILE_FEATURES_FUT)
clusters_mun = pd.read_csv(FILE_CLUSTERS)
coeficientes = pd.read_csv(FILE_COEF)
variables_indice = pd.read_csv(FILE_VARIABLES)["variable"].tolist()
parametros = pd.read_csv(FILE_PARAMETROS).iloc[0].to_dict()
precios = pd.read_csv(FILE_PRECIOS)

print("Features históricas:", features_hist.shape)
print("Features futuras:", features_fut.shape)
print("Variables índice:", len(variables_indice))
print("Municipios con cluster:", clusters_mun.shape)
print("Coeficientes:", coeficientes.shape)
print("Parámetros:", list(parametros.keys())[:8], "...")
print("Precios:", precios.shape)


Features históricas: (450, 505)
Features futuras: (81, 12)
Variables índice: 12
Municipios con cluster: (25, 2)
Coeficientes: (12, 5)
Parámetros: ['version_modelo', 'fecha_calibracion', 'target', 'n_observaciones', 'n_municipios', 'n_clusters', 'n_variables_indice', 'variables_indice'] ...
Precios: (20, 5)


## 4. Validaciones mínimas

In [5]:
# Validaciones de columnas base
cols_hist = ["municipio", "anio", "cluster", "Rendimiento (t/ha)"]
faltantes_hist = [c for c in cols_hist if c not in features_hist.columns]

if faltantes_hist:
    raise ValueError(f"Faltan columnas en features históricas: {faltantes_hist}")

cols_fut_base = ["municipio", "anio"]
faltantes_fut = [c for c in cols_fut_base if c not in features_fut.columns]

if faltantes_fut:
    raise ValueError(f"Faltan columnas en features futuras: {faltantes_fut}")

faltantes_vars_hist = [v for v in variables_indice if v not in features_hist.columns]
faltantes_vars_fut = [v for v in variables_indice if v not in features_fut.columns]

if faltantes_vars_hist:
    raise ValueError(f"Variables del índice faltantes en histórico: {faltantes_vars_hist}")

if faltantes_vars_fut:
    raise ValueError(f"Variables del índice faltantes en futuro: {faltantes_vars_fut}")

if "cluster" not in features_fut.columns:
    features_fut = features_fut.merge(clusters_mun, on="municipio", how="left")

if features_fut["cluster"].isna().sum() > 0:
    raise ValueError("Hay municipios futuros sin clúster asignado.")

features_fut["cluster"] = features_fut["cluster"].astype(int)
features_hist["cluster"] = features_hist["cluster"].astype(int)

print("Validaciones completadas")


ValueError: Variables del índice faltantes en futuro: ['ndvi_min_m12', 'et_potencial_mm_std', 'temp_mean_z_mean_anual', 'ndvi_mean_m05', 'et_potencial_mm_m01', 'precip_max_mm_m07', 'ndvi_min_m01', 'precip_max_mm_m08', 'et_real_mm_std', 'balance_hidrico_mm_min_m01_06', 'ndvi_mean_std', 'ndvi_mean_m02']

## 5. Preparar referencias históricas

Para calcular el índice futuro de forma consistente, se usan las medias y desviaciones históricas por clúster.

Esto replica la lógica del notebook `05_indice_riesgo.ipynb`.


In [ ]:
# Referencias de escalamiento por cluster para las variables del índice
referencias_cluster = {}

for c in sorted(features_hist["cluster"].unique()):
    df_c = features_hist[features_hist["cluster"] == c]

    referencias_cluster[c] = {
        "mean": df_c[variables_indice].mean(),
        "std": df_c[variables_indice].std().replace(0, np.nan)
    }

# Referencias para normalizar el índice raw por cluster
coef_map = coeficientes.set_index("variable")

indice_raw_hist = []

for c in sorted(features_hist["cluster"].unique()):

    idx = features_hist["cluster"] == c
    df_c = features_hist.loc[idx].copy()

    media = referencias_cluster[c]["mean"]
    std = referencias_cluster[c]["std"]

    X_scaled = (df_c[variables_indice] - media) / std
    X_scaled = X_scaled.replace([np.inf, -np.inf], np.nan).fillna(0)

    pesos = coef_map[c].loc[variables_indice]
    raw = X_scaled.mul(pesos, axis=1).sum(axis=1)

    tmp = pd.DataFrame({
        "cluster": c,
        "indice_raw": raw
    })

    indice_raw_hist.append(tmp)

indice_raw_hist = pd.concat(indice_raw_hist, ignore_index=True)

referencias_indice = (
    indice_raw_hist
    .groupby("cluster")["indice_raw"]
    .agg(["mean", "std"])
    .reset_index()
)

referencias_indice


## 6. Preparar precios

In [ ]:
# Detectar columna de precio anual
if "precio_referencia_USD_ton" in precios.columns:
    col_precio = "precio_referencia_USD_ton"
elif "PM30_prom_USD_ton" in precios.columns:
    col_precio = "PM30_prom_USD_ton"
else:
    raise ValueError("No se encontró columna de precio compatible.")

precios_modelo = precios[["anio", col_precio]].copy()
precios_modelo = precios_modelo.rename(columns={col_precio: "precio_USD_ton"})

precios_modelo.tail()


## 7. Funciones del modelo operativo

In [ ]:
def calcular_payout(indice, trigger, limite, exponent):
    if indice >= trigger:
        return 0.0
    elif indice <= limite:
        return 1.0
    else:
        x = (trigger - indice) / (trigger - limite)
        return float(x ** exponent)


def clasificar_riesgo(indice, trigger, umbral_medio):
    if indice <= trigger:
        return "Alto"
    elif indice <= umbral_medio:
        return "Medio"
    else:
        return "Bajo"


def obtener_probabilidad_perdida(indice, df_hist_indice):
    """
    Estima probabilidad empírica de pérdida usando ventanas históricas del índice.
    Busca registros históricos con índice menor o igual al valor consultado.
    """
    subset = df_hist_indice[df_hist_indice["indice_cluster"] <= indice]

    if len(subset) < 10:
        subset = df_hist_indice.nsmallest(30, "indice_cluster")

    return subset["evento_perdida_global"].mean()


def calcular_indice_para_fila(row):
    c = int(row["cluster"])

    media = referencias_cluster[c]["mean"]
    std = referencias_cluster[c]["std"]

    X = row[variables_indice].astype(float)
    X_scaled = (X - media) / std
    X_scaled = X_scaled.replace([np.inf, -np.inf], np.nan).fillna(0)

    pesos = coef_map[c].loc[variables_indice]
    indice_raw = (X_scaled * pesos).sum()

    ref_idx = referencias_indice[referencias_indice["cluster"] == c].iloc[0]

    if ref_idx["std"] == 0 or pd.isna(ref_idx["std"]):
        indice = 0.0
    else:
        indice = (indice_raw - ref_idx["mean"]) / ref_idx["std"]

    return float(indice)


## 8. Reconstruir índice histórico para probabilidad empírica

In [ ]:
# Si existe salida histórica del índice, usarla. Si no, reconstruir lo mínimo.
indice_hist_file = PATH_OUTPUTS / "indice_riesgo_climatico_2007-2024.csv"

if indice_hist_file.exists():
    df_hist_indice = pd.read_csv(indice_hist_file)
else:
    df_hist_indice = features_hist.copy()
    df_hist_indice["indice_cluster"] = df_hist_indice.apply(calcular_indice_para_fila, axis=1)

    umbral_perdida = float(parametros["umbral_perdida_rendimiento"])
    df_hist_indice["evento_perdida_global"] = (
        df_hist_indice["Rendimiento (t/ha)"] <= umbral_perdida
    ).astype(int)

df_hist_indice[["municipio", "anio", "cluster", "indice_cluster", "evento_perdida_global"]].head()


## 9. Función principal de cotización

In [ ]:
def cotizar_seguro(municipio, anio, area_ha):
    """
    Cotiza el seguro indexado para un municipio, año y área cultivada.

    Parámetros
    ----------
    municipio : str
        Nombre del municipio.
    anio : int
        Año de consulta. Puede ser histórico o futuro si existe en las features.
    area_ha : float
        Área cultivada en hectáreas.

    Retorna
    -------
    dict
        Resultado de la cotización.
    """

    municipio = str(municipio)
    anio = int(anio)
    area_ha = float(area_ha)

    # Seleccionar base según año
    if anio <= features_hist["anio"].max():
        base = features_hist.copy()
        historico = True
    else:
        base = features_fut.copy()
        historico = False

    fila = base[
        (base["municipio"] == municipio) &
        (base["anio"] == anio)
    ]

    if fila.empty:
        raise ValueError(f"No hay features disponibles para {municipio} en {anio}.")

    fila = fila.iloc[0].copy()

    # Cluster
    if "cluster" not in fila.index or pd.isna(fila["cluster"]):
        fila_cluster = clusters_mun[clusters_mun["municipio"] == municipio]
        if fila_cluster.empty:
            raise ValueError(f"No hay clúster para el municipio {municipio}.")
        fila["cluster"] = int(fila_cluster["cluster"].iloc[0])

    fila["cluster"] = int(fila["cluster"])

    # Índice y riesgo
    indice = calcular_indice_para_fila(fila)

    trigger = float(parametros["trigger"])
    limite = float(parametros["limite"])
    umbral_medio = float(parametros["umbral_medio_riesgo"])
    exponent = float(parametros["payout_exponent"])

    payout = calcular_payout(indice, trigger, limite, exponent)
    riesgo = clasificar_riesgo(indice, trigger, umbral_medio)
    prob_perdida = obtener_probabilidad_perdida(indice, df_hist_indice)

    # Rendimiento esperado
    if historico and "Rendimiento (t/ha)" in fila.index:
        rendimiento_real = fila["Rendimiento (t/ha)"]
    else:
        rendimiento_real = np.nan

    rendimiento_esperado = (
        features_hist
        .loc[features_hist["municipio"] == municipio, "Rendimiento (t/ha)"]
        .mean()
    )

    if pd.isna(rendimiento_esperado):
        rendimiento_esperado = features_hist["Rendimiento (t/ha)"].mean()

    produccion_esperada_t = rendimiento_esperado * area_ha

    # Precio
    precio_row = precios_modelo[precios_modelo["anio"] == anio]

    if precio_row.empty:
        precio_USD_ton = precios_modelo["precio_USD_ton"].iloc[-1]
    else:
        precio_USD_ton = precio_row["precio_USD_ton"].iloc[0]

    # Montos económicos
    monto_asegurado = produccion_esperada_t * precio_USD_ton

    prima_pura_rate = float(parametros["prima_pura"])
    prima_comercial_rate = float(parametros["prima_comercial"])

    costo_prima_pura = prima_pura_rate * monto_asegurado
    costo_prima_comercial = prima_comercial_rate * monto_asegurado
    indemnizacion_estimada = payout * monto_asegurado

    return {
        "municipio": municipio,
        "anio": anio,
        "area_ha": area_ha,
        "cluster": int(fila["cluster"]),
        "tipo_estimacion": "historica" if historico else "futura",
        "rendimiento_real_t_ha": rendimiento_real,
        "rendimiento_esperado_t_ha": rendimiento_esperado,
        "produccion_esperada_t": produccion_esperada_t,
        "precio_USD_ton": precio_USD_ton,
        "indice_riesgo": indice,
        "nivel_riesgo": riesgo,
        "probabilidad_perdida": prob_perdida,
        "payout": payout,
        "monto_asegurado_USD": monto_asegurado,
        "prima_pura_USD": costo_prima_pura,
        "prima_comercial_USD": costo_prima_comercial,
        "indemnizacion_estimada_USD": indemnizacion_estimada
    }


## 10. Ejemplo de cotización

In [ ]:
# Ejemplo: usar un municipio disponible en features futuras
municipio_demo = features_fut["municipio"].iloc[0]
anio_demo = int(features_fut["anio"].min())
area_demo = 1.0

cotizacion_demo = cotizar_seguro(
    municipio=municipio_demo,
    anio=anio_demo,
    area_ha=area_demo
)

pd.DataFrame([cotizacion_demo])


## 11. Cotizaciones masivas para 2025–2027

In [ ]:
cotizaciones = []

for _, row in features_fut[["municipio", "anio"]].drop_duplicates().iterrows():
    try:
        cot = cotizar_seguro(
            municipio=row["municipio"],
            anio=int(row["anio"]),
            area_ha=1.0
        )
        cotizaciones.append(cot)
    except Exception as e:
        print("Error:", row["municipio"], row["anio"], e)

df_cotizaciones = pd.DataFrame(cotizaciones)

df_cotizaciones.head()


In [ ]:
resumen_cotizaciones = (
    df_cotizaciones
    .groupby(["anio", "nivel_riesgo"])
    .agg(
        n=("municipio", "size"),
        indice_promedio=("indice_riesgo", "mean"),
        prob_perdida_promedio=("probabilidad_perdida", "mean"),
        prima_comercial_promedio_USD_ha=("prima_comercial_USD", "mean"),
        indemnizacion_promedio_USD_ha=("indemnizacion_estimada_USD", "mean")
    )
    .reset_index()
)

resumen_cotizaciones


## 12. Visualización resumen

In [ ]:
plt.figure(figsize=(8, 4))
for riesgo in sorted(df_cotizaciones["nivel_riesgo"].unique()):
    df_r = df_cotizaciones[df_cotizaciones["nivel_riesgo"] == riesgo]
    plt.scatter(df_r["anio"], df_r["prima_comercial_USD"], label=riesgo, alpha=0.7)

plt.xlabel("Año")
plt.ylabel("Prima comercial USD/ha")
plt.title("Prima comercial estimada por año y nivel de riesgo")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(8, 4))
for riesgo in sorted(df_cotizaciones["nivel_riesgo"].unique()):
    df_r = df_cotizaciones[df_cotizaciones["nivel_riesgo"] == riesgo]
    plt.scatter(df_r["indice_riesgo"], df_r["probabilidad_perdida"], label=riesgo, alpha=0.7)

plt.xlabel("Índice de riesgo")
plt.ylabel("Probabilidad empírica de pérdida")
plt.title("Índice vs probabilidad de pérdida")
plt.legend()
plt.grid(True)
plt.show()


## 13. Guardar resultados

In [ ]:
df_cotizaciones.to_csv(OUTPUT_COTIZACIONES, index=False)

OUTPUT_RESUMEN_COT = PATH_OUTPUTS / "resumen_cotizaciones_modelo_seguro.csv"
resumen_cotizaciones.to_csv(OUTPUT_RESUMEN_COT, index=False)

print("Archivos guardados:")
print(OUTPUT_COTIZACIONES)
print(OUTPUT_RESUMEN_COT)


## 14. Siguiente paso: implementación API

Este notebook define la lógica que debe migrarse a archivos `.py`:

```text
src/
├── data_loader.py
├── risk_index.py
├── insurance_model.py
└── api.py
```

La función central que debe exponerse en la API es:

```python
cotizar_seguro(municipio, anio, area_ha)
```

con salida JSON equivalente a la tabla de cotización generada en este notebook.
